# 🎯 Module 13 · 06 — Word Embeddings

> **Stage:** Machine Learning → NLP  
> **Level:** Intermediate  
> **Module:** 13 — NLP Basics / 06 — Word Embeddings  
> **Date:** 30 January 2026  
> **Previous:** 05 — TF-IDF  
> **Next:** 07 — Text Classification

---

**Word embeddings** represent words as **dense numerical vectors** — where words with similar meanings or usage have similar representations.

In this notebook we'll cover:

- Why embeddings exist
- **Sparse vs dense** representations
- The **distributional hypothesis**
- **Word2Vec** (CBOW & Skip-Gram)
- **GloVe**
- **FastText**
- Embedding dimensions
- **Cosine similarity** & semantic relationships
- **Document embeddings**
- Pretrained embeddings
- Applications
- Static embedding limitations
- TF-IDF vs embeddings

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain what word embeddings are
- Distinguish sparse vs dense representations
- Explain the distributional hypothesis
- Describe Word2Vec (**CBOW** and **Skip-Gram**)
- Explain **GloVe** and **FastText**
- Compute **cosine similarity** between word vectors
- Build a simple **document embedding** by averaging
- Explain **static embedding limitations**
- Contrast **TF-IDF vs embeddings**
- Explain why context matters

---
## ⚙️ 0. Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
print('✅ Setup ready.')

In [ ]:
# A tiny training corpus we'll use to demonstrate embeddings
sentences = [
    ['i', 'love', 'python'],
    ['i', 'love', 'machine', 'learning'],
    ['python', 'is', 'powerful'],
    ['machine', 'learning', 'is', 'useful'],
    ['i', 'love', 'deep', 'learning'],
    ['deep', 'learning', 'is', 'powerful'],
    ['python', 'and', 'machine', 'learning'],
    ['i', 'enjoy', 'coding', 'in', 'python'],
]

for s in sentences:
    print(' ', s)

---
## 🧠 1. Why Do We Need Embeddings?

BoW and TF-IDF represent words based on **occurrence**.

They treat:

```text
car
automobile
vehicle
```

as **separate, unrelated** features.

**Word embeddings** capture relationships from how words **appear in context**:

```text
car        → [0.20,  0.70, -0.10, ...]
automobile → [0.21,  0.69, -0.08, ...]   ← close to "car"
banana     → [-0.40, 0.10,  0.80, ...]   ← far away
```

Related words → **closer** vectors.

---
## ⚖️ 2. Sparse vs Dense Representation

### Sparse (BoW / TF-IDF)

```text
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, ...]
```

Most values are zero.

### Dense (Embeddings)

```text
[0.21, -0.43, 0.71, 0.18, 0.52]
```

Most values are **non-zero**.

| Feature | Sparse | Dense |
| --- | --- | --- |
| Examples | BoW, TF-IDF | Word2Vec, GloVe |
| Many zeros | ✅ | Usually ❌ |
| Dimensions | Very large | Smaller |
| Semantic info | Limited | Better |
| Representation | Word occurrence | Learned vector |

In [ ]:
# Visual comparison: sparse vs dense
sparse_vec = np.array([0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0.4, 0, 0])
dense_vec  = np.random.randn(14)

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].bar(range(len(sparse_vec)), sparse_vec)
axes[0].set_title(f'Sparse — zeros: {(sparse_vec == 0).sum()}/{len(sparse_vec)}')
axes[0].grid(alpha=0.3, axis='y')

axes[1].bar(range(len(dense_vec)), dense_vec)
axes[1].set_title(f'Dense — zeros: {(dense_vec == 0).sum()}/{len(dense_vec)}')
axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

---
## 📜 3. The Distributional Hypothesis

> **Words that occur in similar contexts tend to have related meanings.**

### Example

```text
"I drink hot coffee."
"I drink hot tea."
```

`coffee` and `tea` appear in **similar contexts**.

So their embeddings should be **close** to each other.

This is the foundation of **Word2Vec, GloVe, and FastText**.

---
## 🔤 4. Word2Vec

**Word2Vec** learns word vectors from **surrounding words** in a large corpus.

Two architectures:

```text
CBOW
Skip-Gram
```

Both are **shallow neural networks** trained on a simple prediction task.

---
## 🎯 5. CBOW — Continuous Bag of Words

**CBOW** predicts a **target word** from its **surrounding context**.

### Example

```text
"The cat drinks milk"
```

Target: `drinks`  
Context: `cat`, `milk`

```text
Context Words
      ↓
    Model
      ↓
Target Word
```

- Faster to train
- Works well for frequent words

---
## 🎯 6. Skip-Gram

**Skip-Gram** uses a **target word** to predict **surrounding context words**.

### Example

```text
"The cat drinks milk"
```

Target: `cat`  
Context: `the`, `drinks`

```text
Target Word
     ↓
   Model
     ↓
Context Words
```

### CBOW vs Skip-Gram

```text
CBOW        Context → Target
Skip-Gram   Target  → Context
```

> Skip-Gram tends to work better for **rare words**.

---
## 💡 7. Word2Vec Intuition

From a corpus like:

```text
"The cat drinks milk"
"The dog drinks water"
"The cat likes fish"
"The dog likes meat"
```

The model observes **which words appear near each other**.

Over time, words with **similar contexts** end up **close** in vector space:

```text
          cat
           ●
          / \
         /   \
      dog     milk
       ●
```

(Illustration only — real embedding spaces have 50–300+ dimensions.)

---
## 🌍 8. GloVe

**GloVe** — Global Vectors for Word Representation.

Uses **global word co-occurrence statistics**:

```text
Large Corpus
     ↓
Word Co-occurrence Information
     ↓
Learning Algorithm
     ↓
Word Vectors
```

### Word2Vec vs GloVe

- Word2Vec = **local context windows**
- GloVe = **global co-occurrence matrix**

Both produce useful dense representations.

---
## ⚡ 9. FastText

**FastText** extends embeddings with **subword information**.

Instead of treating a word as one unit, it uses **character n-grams**.

### Useful for

- Rare words
- Misspellings
- Morphologically rich languages
- Words not seen during training

### Example

`playing` → character pieces like `pla`, `lay`, `ayi`, `yin`, `ing`

> Even unseen words can get reasonable vectors.

---
## 📏 10. Embedding Dimensions

Typical sizes:

```text
50, 100, 200, 300
```

A 100-D embedding:

```text
word → [0.21, -0.43, 0.71, ...  100 values  ...]
```

### ⚠️ Dimensions are **not** human-interpretable

We **cannot** say:

```text
dim 1 = happiness
dim 2 = size
```

They are **learned features** without a simple human label.

---
## 📐 11. Word Similarity (Cosine)

Once words are vectors, we compare them:

```text
Word A → Vector A
Word B → Vector B
       ↓
Cosine Similarity
       ↓
Similarity Score
```

Pairs with **high similarity**:

```text
king ↔ queen
car ↔ automobile
doctor ↔ physician
```

In [ ]:
# Cosine similarity between vectors
def cosine(a, b):
    a, b = np.asarray(a), np.asarray(b)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else 0.0

# Fake embeddings for demo (real ones come from training)
np.random.seed(42)
vecs = {
    'king':       np.array([0.9, 0.1, 0.8]),
    'queen':      np.array([0.85, 0.12, 0.79]),
    'car':        np.array([-0.7, 0.4, -0.2]),
    'automobile': np.array([-0.68, 0.42, -0.18]),
    'banana':     np.array([0.1, -0.9, 0.3]),
}

print(f'{"Pair":<25} {"Cosine":>8}')
print('-' * 34)
for a, b in [('king','queen'), ('car','automobile'), ('king','banana'), ('car','banana')]:
    print(f'{a+" ↔ "+b:<25} {cosine(vecs[a], vecs[b]):>8.3f}')

---
## 🔗 12. Semantic Relationships

Some embedding spaces exhibit **linear relationships**:

```text
king - man + woman ≈ queen
```

⚠️ This is an **observation** about certain learned spaces — **not a universal rule**.

### Quality depends on:

- Training data
- Algorithm
- Vocabulary size
- Hyperparameters
- Domain

---
## 📄 13. Document Embeddings

Word embeddings represent **individual words**.

Many tasks need a **document vector** — combine word vectors by **averaging** (or another aggregation).

```text
Document
   ↓
Words
   ↓
Word Embeddings
   ↓
Average / Aggregate
   ↓
Document Vector
```

### ⚠️ Limitation

Averaging **loses word order and much of the context**.

In [ ]:
# Fake word vectors to demonstrate document embedding
np.random.seed(42)
fake_vocab = ['i', 'love', 'python', 'machine', 'learning', 'deep', 'coding']
fake_embed = {w: np.random.randn(5) for w in fake_vocab}

def document_vector(doc):
    vecs = [fake_embed[w] for w in doc.lower().split() if w in fake_embed]
    return np.mean(vecs, axis=0) if vecs else np.zeros(5)

for doc in ["i love python", "i love machine learning", "i love deep learning"]:
    print(f'{doc:<30} -> {document_vector(doc).round(3)}')

# Cosine similarity between document vectors
d1 = document_vector("i love python")
d2 = document_vector("i love machine learning")
d3 = document_vector("i love deep learning")

print()
print(f'Cosine(d1, d2) = {cosine(d1, d2):.3f}')
print(f'Cosine(d1, d3) = {cosine(d1, d3):.3f}')
print(f'Cosine(d2, d3) = {cosine(d2, d3):.3f}')

---
## 🌍 14. Embeddings for NLP Tasks

### Text Similarity

```text
Doc A → Embedding
Doc B → Embedding
         ↓
       Compare
```

### Search
Represent queries and documents as vectors, compare.

### Classification
Use embeddings as **input features** for a classifier.

### Clustering
Group documents or words by their vector proximity.

### Recommendation
Compare item or document representations by **semantic similarity**.

---
## 📦 15. Pretrained Embeddings

We don't always need to train embeddings from scratch.

Pretrained options:

```text
Word2Vec
GloVe
FastText
```

### Workflow

```text
Large External Corpus
        ↓
Pretrained Embeddings
        ↓
Your NLP Project
        ↓
Similarity / Features / Model
```

> Useful when your own dataset is **small**.

---
## ⚠️ 16. Static Embedding Limitation — Context

Traditional embeddings give **one vector per word**.

But:

```text
"I deposited money at the bank."
"I sat beside the river bank."
```

The word **bank** has **different meanings**.

A static embedding cannot distinguish these.

### Solved by (later):

```text
Contextual Embeddings
Transformers
BERT
GPT-style models
```

These produce representations that **depend on the sentence**.

---
## ⚖️ 17. Word Embeddings vs TF-IDF

| Feature | TF-IDF | Word Embeddings |
| --- | --- | --- |
| Representation | Sparse | Dense |
| Based on word frequency | ✅ | Not primarily |
| Semantic relationships | Limited | ✅ Better |
| Context | Limited | Depends on method |
| Dimensions | Very large | Smaller |
| Interpretability | Relatively high | Lower |
| Classical ML baseline | ✅ Excellent | Can be useful |

### Simple mental model

```text
TF-IDF      -> How important is this word in this document?
Embeddings  -> How is this word represented in relation to other words?
```

---
## 🧪 18. Practical Example — Train Word2Vec

```python
from gensim.models import Word2Vec

model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)
model.wv['python']            # vector
model.wv.most_similar('python')  # similar words
```

In [ ]:
# Try to train a real Word2Vec model (install: pip install gensim)
try:
    from gensim.models import Word2Vec

    model = Word2Vec(
        sentences,
        vector_size=20,
        window=3,
        min_count=1,
        sg=1,   # 1 = Skip-Gram, 0 = CBOW
        workers=1,
        seed=42,
        epochs=200,
    )

    print('Vocabulary size:', len(model.wv.index_to_key))
    print()
    print('Vector for "python" (first 5 dims):')
    print(model.wv['python'][:5].round(3))
    print()
    print('Most similar to "python":')
    for word, sim in model.wv.most_similar('python', topn=3):
        print(f'  {word:<10} {sim:.3f}')

    print()
    print('Most similar to "learning":')
    for word, sim in model.wv.most_similar('learning', topn=3):
        print(f'  {word:<10} {sim:.3f}')

except ImportError:
    print('Gensim not installed. Run: pip install gensim')

---
## 🌐 19. Real-World Considerations

Embedding quality depends heavily on the **training corpus**:

| Corpus | What it captures |
| --- | --- |
| Technical documents | technical terminology |
| Social media | slang, informal language |
| Medical text | medical relationships |
| Wikipedia | general-purpose facts |

### 🧭 Golden Rule

> **Embedding quality is strongly influenced by the data and domain used to train them.**

---
## 🧠 20. Mental Model

```text
Raw Text
   ↓
Tokens
   ↓
Embedding Model
   ↓
Dense Vectors
   ↓
Similarity / Classification / Clustering
```

### Classical embedding approaches

```text
Word2Vec
 ├── CBOW
 └── Skip-Gram

GloVe

FastText
```

### Key idea

> **Word embeddings transform words into dense vectors where useful relationships can be learned from how words occur in language.**

---
## 🏋️ 21. Hands-On Exercises

### Exercise 1 — Sparse vs dense
For a 10,000-word vocabulary, how many values are in:
- a one-hot vector
- a 100-D embedding
Which is **memory-friendly**?

### Exercise 2 — CBOW vs Skip-Gram
Given:
```text
"the dog ate the food"
```
Write the CBOW and Skip-Gram setups for the word `ate`.

### Exercise 3 — Cosine similarity
Compute cosine similarity between:
```python
a = [1, 0, 0]
b = [0.9, 0.1, 0]
c = [0, 1, 0]
```
Which pair is most similar?

### Exercise 4 — Train Word2Vec
Train a Word2Vec model on the sample sentences.  
Compare `sg=1` (Skip-Gram) vs `sg=0` (CBOW).  
Which gives better neighbors?

### Exercise 5 — Document embeddings
Average word vectors for:
```text
"I love machine learning"
"I love deep learning"
"I hate traffic"
```
Which two are most similar by cosine?

### Exercise 6 — Static limitation
Explain why **bank** gets the same vector in "river bank" and "bank account".  
How would a contextual model solve this?

---
## 🎤 22. Interview Questions

1. **What is a word embedding?**  
   A dense numerical vector representation of a word.

2. **Sparse vs dense?**  
   Sparse = mostly zeros (BoW, TF-IDF). Dense = mostly non-zeros (embeddings).

3. **Why are embeddings useful?**  
   They capture semantic relationships that sparse representations miss.

4. **What is the distributional hypothesis?**  
   Words in similar contexts tend to have related meanings.

5. **What is Word2Vec?**  
   A neural method that learns word vectors from surrounding context words.

6. **CBOW vs Skip-Gram?**  
   CBOW predicts a target from context; Skip-Gram predicts context from a target.

7. **What is GloVe?**  
   An embedding method using global word co-occurrence statistics.

8. **What is FastText?**  
   An extension of Word2Vec that uses character n-grams for subword information.

9. **How big are typical embeddings?**  
   50–300 dimensions.

10. **What is cosine similarity?**  
    A measure of angle between vectors — the standard way to compare embeddings.

11. **What is a document embedding?**  
    A single vector representing a document — often by averaging word vectors.

12. **Why use pretrained embeddings?**  
    To leverage large external corpora when your own dataset is small.

13. **Main limitation of static embeddings?**  
    One vector per word — they cannot distinguish different meanings by context.

14. **TF-IDF vs embeddings?**  
    TF-IDF weights words by occurrence; embeddings capture semantics in dense vectors.

15. **What replaced static embeddings?**  
    Contextual embeddings (ELMo, BERT, GPT-style models).

---
## 🏁 23. Key Takeaways

### You now understand

- What word embeddings are
- Sparse vs dense
- The distributional hypothesis
- **Word2Vec** — CBOW & Skip-Gram
- **GloVe**
- **FastText**
- Embedding dimensions
- **Cosine similarity**
- Semantic relationships
- **Document embeddings** via averaging
- Pretrained embeddings
- NLP applications of embeddings
- Static embedding limitations
- **TF-IDF vs embeddings**
- Why context matters

### The progression

```text
One-Hot
  ↓
Bag of Words
  ↓
TF-IDF
  ↓
Word Embeddings (Word2Vec, GloVe, FastText)
  ↓
Contextual Embeddings (BERT, GPT — later)
```

---

### 🔗 What's Next?

**07 — Text Classification**

We combine **text representations + classical ML models** to build practical text classifiers.